# 01 · Spatial-to-H&E registration

Read matched histology and spatial transcriptomic data, align their signals, and save cell coordinates in native H&E pixels.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/02_he_prediction").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "he_reference_v2")
OUT = ROOT / "outputs/he_prediction" / RUN_ID
HE_IMAGE = ROOT / "data/he_slides/proseg1.tif"
INPUT_H5AD = ROOT / "data/processed/proseg_1.h5ad"
OUTPUT_H5AD = OUT / "alignment/proseg_1_aligned.h5ad"
HE_SCALE = 0.1
DX = 30
GENERATIONS = 1000
ITERATIONS = 2000
REPEATS = 2
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc._io import require_files
from spagc.histology.registration import (
    export_registration,
    optimize_registration,
    prepare_registration,
)

Declare the real H&E image and processed cell object and check that both exist.

In [ ]:
require_files(HE_IMAGE, INPUT_H5AD)

## 2. Read images and form the registration signals

Read native H&E and cell data. Form a downsampled Sobel image signal and a rasterized expression signal in their declared coordinate frames.

Keep the loaded cells, image signals and image scale together during registration.

Read the H&E image and processed cells, form the image-gradient and expression rasters, and prepare registration inputs.

In [ ]:
prepared = prepare_registration(
    HE_IMAGE, INPUT_H5AD, image_scale=HE_SCALE, raster_spacing=DX
)
print(
    {
        "cells": prepared.cells.n_obs,
        "he_raster": prepared.image_signal.shape,
        "st_raster": prepared.st_signal.shape,
        "signal_source": prepared.signal_source,
    }
)

## 3. Fit the matched-section registration

Run GALA on the two signals using the declared optimization settings.

Fit the matched-section GALA transformation using the declared search settings.

In [ ]:
transform = optimize_registration(
    prepared, generations=GENERATIONS, iterations=ITERATIONS, repeats=REPEATS, seed=SEED
)

## 4. Save aligned cells and the registration overlay

Transform cell centroids into native image pixels, applying the downsampling scale once. Save aligned AnnData and an overlay for visual inspection.

Project cell coordinates back to native H&E pixels and save aligned cells and an overlay.

In [ ]:
aligned = export_registration(
    prepared,
    transform,
    OUTPUT_H5AD,
    image_path=HE_IMAGE,
    cells_path=INPUT_H5AD,
    raster_spacing=DX,
    seed=SEED,
)
print(OUTPUT_H5AD)